In [1]:
import copy
import importlib
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
from scipy.spatial import cKDTree
import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2/model"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp 
import svensemble.bndbpconv as bndc
import svensemble.utils as sv_utils

importlib.reload(vcfp)
importlib.reload(bndc)
importlib.reload(sv_utils)

<module 'svensemble.utils' from '/vast/ai_projects/Papenfuss_SVEnsemble2/model/svensemble/utils.py'>

In [2]:
pd.set_option('display.max_columns', None)

The current idea is:

- split the reference events into separate breakends (DELs have 1 breakpoint with 2 breakends, do others later)
- for each breakend, find the closest breakend in the gridss callset
- calculate the overlap fraction
- keep only those with > 80% overlap
- train model to predic correct breakends
- assemble into simple events later (before benchmark)


In [4]:
gridss_callset, gridss_meta = vcfp.parse_vcf(
    "data/test_gridss_output.NA12878.vcf",
    line_limit = None, 
    output_meta=True
)

Processed 0 entries                 Elapsed time: 1.9073486328125e-05 seconds.
Processed 100000 entries                 Elapsed time: 4.755464553833008 seconds.
Processed 200000 entries                 Elapsed time: 9.598597049713135 seconds.
Processed 300000 entries                 Elapsed time: 14.491267919540405 seconds.
Processed 400000 entries                 Elapsed time: 19.26408338546753 seconds.
Processed 500000 entries                 Elapsed time: 24.09964609146118 seconds.
Processed 600000 entries                 Elapsed time: 28.953471422195435 seconds.
Processed 700000 entries                 Elapsed time: 33.7425742149353 seconds.
Processed 800000 entries                 Elapsed time: 38.512430906295776 seconds.
Processed 900000 entries                 Elapsed time: 43.37598657608032 seconds.
Processed 1000000 entries                 Elapsed time: 48.11571741104126 seconds.
Finished processing. Processed 1007409 entries out of 1007410. Elapsed time: 73.4600076675415 seco

In [5]:
ref_callset, ref_meta = vcfp.parse_vcf(
    'data/gnomad.v4.1.sv.sites.vcf.gz',
    line_limit = None, skip_format=True, 
    info_limit_to=['SVTYPE', 'SVLEN', 'END', 'CHR2', 'EVIDENCE'],
    output_meta=True
)


Processed 0 entries                 Elapsed time: 0.006846427917480469 seconds.
Processed 100000 entries                 Elapsed time: 5.69795823097229 seconds.
Processed 200000 entries                 Elapsed time: 11.436517715454102 seconds.
Processed 300000 entries                 Elapsed time: 17.061073303222656 seconds.
Processed 400000 entries                 Elapsed time: 22.645848512649536 seconds.
Processed 500000 entries                 Elapsed time: 28.222749948501587 seconds.
Processed 600000 entries                 Elapsed time: 33.80459237098694 seconds.
Processed 700000 entries                 Elapsed time: 39.40978789329529 seconds.
Processed 800000 entries                 Elapsed time: 45.029358863830566 seconds.
Processed 900000 entries                 Elapsed time: 50.616684913635254 seconds.
Processed 1000000 entries                 Elapsed time: 56.2084755897522 seconds.
Processed 1100000 entries                 Elapsed time: 61.79177904129028 seconds.
Processed 12

In [6]:
ref_callset_copy = copy.deepcopy(ref_callset)
ref_meta_copy = copy.deepcopy(ref_meta)

viable_svtypes = ['DEL', 'DUP', 'INS', 'INV']

ref_callset_copy = ref_callset_copy[ref_callset_copy['INFO_SVTYPE'].isin(viable_svtypes)]
ref_callset_copy = ref_callset_copy[ref_callset_copy['FILTER_PASS'].astype(bool)]

sv_utils.split_evidence_col(ref_callset_copy, ref_meta_copy)
sv_utils.delete_mult_val_cols(ref_callset_copy, ref_meta_copy, exceptions=['INFO_EVIDENCE'], vocal=True)
sv_utils.convert_col_types(ref_callset_copy, ref_meta_copy)

ref_callset_copy.loc[:, 'INFO_EVENT'] = ref_callset_copy['ID']

Dropping entries ['FILTER_PASS', 'FILTER_FAIL_MANUAL_REVIEW', 'FILTER_HIGH_NCR', 'FILTER_IGH_MHC_OVERLAP', 'FILTER_LOWQUAL_WHAM_SR_DEL', 'FILTER_MULTIALLELIC', 'FILTER_OUTLIER_SAMPLE_ENRICHED', 'FILTER_REDUNDANT_LG_CNV', 'FILTER_REFERENCE_ARTIFACT', 'FILTER_UNRESOLVED', 'INFO_EVIDENCE'] since they contain multiple values


Now we separate event entries into breakend entries

Each event type is handled separately

Only simple events are considered for now

4 different combinations of partner position/direction are possible:

- N[PARTNER[ = partner extending after breakpoint - ENCODE AS 1

- ]PARTNER]N = partner extending before breakpoint - ENCODE AS 2

- N]PARTNER] = reverse complement of partner extending after breakpoint - ENCODE AS 3

- [PARTNER[N = reverse complement of partner extending before breakpoint - ENCODE AS 4

**Deletions are easiest:**

Each deletion = 1 breakend, 2 breakpoints

bp1: CHROM:POS-1 N[CHR2:END+1[ bnd_pos_encoding = 1

bp2: CHR2:END+1 ]CHROM:POS-1]N bnd_pos_encoding = 2

**Inversions are slightly more complicated, but all breakends are still resolved:**

Each inversion = 2 breakends, 4 breakpoints

bp1: CHROM:POS-1 N]CHR2:END] bnd_pos_encoding = 3

bp2: CHROM:POS [CHR2:END+1[N bnd_pos_encoding = 4

bp3: CHR2:END N]CHROM:POS-1] bnd_pos_encoding = 3

bp4: CHR2:END+1 [CHROM:POS[N bnd_pos_encoding = 4

**DUPs are more complicated:**

Each DUP = 2 breakends

bp1: CHR2:END N[CHROM:POS[ bnd_pos_encoding = 1

bp2: CHROM:POS ]CHR2:END]N bnd_pos_encoding = 2

Difference from DELs - positions of breakends are reversed (bp2 goes BEFORE bp1)

INTERESTING - we could assume tandem duplication and have 2 potential breakend partners to the right and left of the duplicated sequence

some weighting scheme would need to be used with these partners, since there would be 0.5 likelihood each is the correct one.

for now - ignore this

**Finally, INSs are tricky too:**

If it is a novel insertion, it will consist of 2 unresolved breakends

if it is an insertion from somewhere else, the breakends will be resolved

For now we will only consider novel insertions

bp1: CHROM:POS-1 N[CHR2:END+1[ bnd_pos_encoding = 1

bp2: CHR2:END+1 ]CHROM:POS-1]N bnd_pos_encoding = 2

In [7]:
ref_callset_bnds = bndc.event_to_bnd(ref_callset_copy, add_encoding=True)
ref_callset_bnds

,CHROM,POS,ID,REF,ALT,QUAL,INFO_CHR2,INFO_END,INFO_SVLEN,INFO_SVTYPE,INFO_EVIDENCE_SR,INFO_EVIDENCE_PE,INFO_EVIDENCE_RD,INFO_EVIDENCE_BEST,INFO_EVENT,INFO_MATEID,bnd_pos_encoding
10,chr1,39999,gnomAD-SV_v3_DEL_chr1_b26f63f7_bp1,N,N[chr1:47001[,145.0,chr1,47000,7000,DEL,0,0,1,RD,gnomAD-SV_v3_DEL_chr1_b26f63f7,gnomAD-SV_v3_DEL_chr1_b26f63f7_bp2,1
16,chr1,59100,gnomAD-SV_v3_DEL_chr1_b0fb90d2_bp1,N,N[chr1:59203[,402.0,chr1,59202,101,DEL,1,0,0,SR,gnomAD-SV_v3_DEL_chr1_b0fb90d2,gnomAD-SV_v3_DEL_chr1_b0fb90d2_bp2,1
18,chr1,62398,gnomAD-SV_v3_DEL_chr1_7ef5c6e8_bp1,N,N[chr1:62490[,588.0,chr1,62489,90,DEL,1,0,0,SR,gnomAD-SV_v3_DEL_chr1_7ef5c6e8,gnomAD-SV_v3_DEL_chr1_7ef5c6e8_bp2,1
22,chr1,66128,gnomAD-SV_v3_DEL_chr1_ebfa4400_bp1,N,N[chr1:66614[,1.0,chr1,66613,484,DEL,1,0,0,SR,gnomAD-SV_v3_DEL_chr1_ebfa4400,gnomAD-SV_v3_DEL_chr1_ebfa4400_bp2,1
26,chr1,66281,gnomAD-SV_v3_DEL_chr1_69f73472_bp1,N,N[chr1:66571[,1.0,chr1,66570,288,DEL,1,0,0,SR,gnomAD-SV_v3_DEL_chr1_69f73472,gnomAD-SV_v3_DEL_chr1_69f73472_bp2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2152030,chrY,26469747,gnomAD-SV_v3_INS_chrY_afd58107_bp2,N,]chrY:26469746]N,692.0,chrY,26469746,281,INS,1,0,0,SR,gnomAD-SV_v3_INS_chrY_afd58107,gnomAD-SV_v3_INS_chrY_afd58107_bp1,2
2152055,chrY,26599416,gnomAD-SV_v3_INS_chrY_59b97d4a_bp2,N,]chrY:26599415]N,1.0,chrY,26599415,64,INS,1,0,0,SR,gnomAD-SV_v3_INS_chrY_59b97d4a,gnomAD-SV_v3_INS_chrY_59b97d4a_bp1,2
2154380,chrY,56851318,gnomAD-SV_v3_INS_chrY_8ccd65b5_bp2,N,]chrY:56851317]N,808.0,chrY,56851317,69,INS,1,0,0,SR,gnomAD-SV_v3_INS_chrY_8ccd65b5,gnomAD-SV_v3_INS_chrY_8ccd65b5_bp1,2
2154402,chrY,56857192,gnomAD-SV_v3_INS_chrY_fa3d9b4f_bp2,N,]chrY:56857191]N,330.0,chrY,56857191,281,INS,1,0,0,SR,gnomAD-SV_v3_INS_chrY_fa3d9b4f,gnomAD-SV_v3_INS_chrY_fa3d9b4f_bp1,2


In [8]:
list(ref_callset_bnds['CHROM'].unique())

['chr1',
 'chr2',
 'chr3',
 'chr4',
 'chr5',
 'chr6',
 'chr7',
 'chr8',
 'chr9',
 'chr10',
 'chr11',
 'chr12',
 'chr13',
 'chr14',
 'chr15',
 'chr16',
 'chr17',
 'chr18',
 'chr19',
 'chr20',
 'chr21',
 'chr22',
 'chrX',
 'chrY',
 '0']

In [9]:
ref_callset_bnds['INFO_SVTYPE'].value_counts()

INFO_SVTYPE
DEL    1255894
INS     592368
DUP     517764
INV       8740
Name: count, dtype: int64

In [10]:
evidence_margins = {
    'SR': 50,
    'PE': 100,
    'RD': 500
}
match_margins = ref_callset_bnds['INFO_EVIDENCE_BEST'].map(evidence_margins)

ref_callset_bnds.loc[:, 'INFO_CIPOS_l'] = - match_margins
ref_callset_bnds.loc[:, 'INFO_CIPOS_r'] = match_margins

In [11]:
ref_callset_bps = bndc.bnd_to_bp(ref_callset_bnds, force_diff_cols=['INFO_CIPOS_l', 'INFO_CIPOS_r'])
ref_callset_bps.loc[:, 'bptype'] = ref_callset_bps['bnd_pos_encoding_bnd1'].astype(str) + ref_callset_bps['bnd_pos_encoding_bnd2'].astype(str)

{'chr1': 0, 'chr2': 1, 'chr3': 2, 'chr4': 3, 'chr5': 4, 'chr6': 5, 'chr7': 6, 'chr8': 7, 'chr9': 8, 'chr10': 9, 'chr11': 10, 'chr12': 11, 'chr13': 12, 'chr14': 13, 'chr15': 14, 'chr16': 15, 'chr17': 16, 'chr18': 17, 'chr19': 18, 'chr20': 19, 'chr21': 20, 'chr22': 21, 'chrX': 22, 'chrY': 23, 'chrM': 24, '0': 25}


In [12]:
ref_callset_bps

,CHROM_bnd1,POS_bnd1,ID_bnd1,REF,ALT_bnd1,QUAL_bnd1,INFO_CHR2,INFO_END,INFO_SVLEN,INFO_SVTYPE,INFO_EVIDENCE_SR,INFO_EVIDENCE_PE,INFO_EVIDENCE_RD,INFO_EVIDENCE_BEST,INFO_EVENT,INFO_MATEID_bnd1,bnd_pos_encoding_bnd1,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,CHROM_bnd2,POS_bnd2,ID_bnd2,ALT_bnd2,QUAL_bnd2,INFO_MATEID_bnd2,bnd_pos_encoding_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,bptype
0,chr12,970833,GD_12p13.33__DEL_bp1,N,N[0:1237306[,NaN,0,1237305,266471,DEL,0,0,1,RD,GD_12p13.33__DEL,GD_12p13.33__DEL_bp2,1,-500,500,0,1237306,GD_12p13.33__DEL_bp2,]0:1237305]N,NaN,GD_12p13.33__DEL_bp1,2,-500,500,12
1,chr15,22782170,GD_15q11-q13-BP1-BP3__DUP_bp2,N,]0:28134728]N,NaN,0,28134728,5352558,DUP,0,0,1,RD,GD_15q11-q13-BP1-BP3__DUP,GD_15q11-q13-BP1-BP3__DUP_bp1,2,-500,500,0,28134728,GD_15q11-q13-BP1-BP3__DUP_bp1,N[0:28134728[,NaN,GD_15q11-q13-BP1-BP3__DUP_bp2,1,-500,500,21
2,chr15,23502848,GD_15q11-q13-BP2-BP3__DUP_bp2,N,]0:28181259]N,NaN,0,28181259,4678411,DUP,0,0,1,RD,GD_15q11-q13-BP2-BP3__DUP,GD_15q11-q13-BP2-BP3__DUP_bp1,2,-500,500,0,28181259,GD_15q11-q13-BP2-BP3__DUP_bp1,N[0:28181259[,NaN,GD_15q11-q13-BP2-BP3__DUP_bp2,1,-500,500,21
3,chr15,22784509,GD_15q11.2-BP1-BP2__DUP_bp2,N,]0:23074432]N,NaN,0,23074432,289923,DUP,0,0,1,RD,GD_15q11.2-BP1-BP2__DUP,GD_15q11.2-BP1-BP2__DUP_bp1,2,-500,500,0,23074432,GD_15q11.2-BP1-BP2__DUP_bp1,N[0:23074432[,NaN,GD_15q11.2-BP1-BP2__DUP_bp2,1,-500,500,21
4,chr15,30844900,GD_15q13.3-BP4-BP5__DEL_bp1,N,N[0:32153208[,NaN,0,32153207,1308306,DEL,0,0,1,RD,GD_15q13.3-BP4-BP5__DEL,GD_15q13.3-BP4-BP5__DEL_bp2,1,-500,500,0,32153208,GD_15q13.3-BP4-BP5__DEL_bp2,]0:32153207]N,NaN,GD_15q13.3-BP4-BP5__DEL_bp1,2,-500,500,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187378,chrX,120004063,gnomAD-SV_v3_INV_chrX_f10e1fc7_bp2,N,[chrX:120004661[N,803.0,chrX,120004660,597,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_f10e1fc7_e2,gnomAD-SV_v3_INV_chrX_f10e1fc7_e2_bp4,4,-50,50,chrX,120004661,gnomAD-SV_v3_INV_chrX_f10e1fc7_bp4,[chrX:120004661[N,803.0,gnomAD-SV_v3_INV_chrX_f10e1fc7_e2_bp2,4,-50,50,44
1187379,chrX,151099145,gnomAD-SV_v3_INV_chrX_f5492f1b_bp1,N,N]chrX:151099241],395.0,chrX,151099241,95,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_f5492f1b_e1,gnomAD-SV_v3_INV_chrX_f5492f1b_e1_bp3,3,-50,50,chrX,151099241,gnomAD-SV_v3_INV_chrX_f5492f1b_bp3,N]chrX:151099240],395.0,gnomAD-SV_v3_INV_chrX_f5492f1b_e1_bp1,3,-50,50,33
1187380,chrX,151099146,gnomAD-SV_v3_INV_chrX_f5492f1b_bp2,N,[chrX:151099242[N,395.0,chrX,151099241,95,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_f5492f1b_e2,gnomAD-SV_v3_INV_chrX_f5492f1b_e2_bp4,4,-50,50,chrX,151099242,gnomAD-SV_v3_INV_chrX_f5492f1b_bp4,[chrX:151099242[N,395.0,gnomAD-SV_v3_INV_chrX_f5492f1b_e2_bp2,4,-50,50,44
1187381,chrX,19842428,gnomAD-SV_v3_INV_chrX_fbd8f758_bp1,N,N]chrX:19843163],868.0,chrX,19843163,734,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_fbd8f758_e1,gnomAD-SV_v3_INV_chrX_fbd8f758_e1_bp3,3,-50,50,chrX,19843163,gnomAD-SV_v3_INV_chrX_fbd8f758_bp3,N]chrX:19843162],868.0,gnomAD-SV_v3_INV_chrX_fbd8f758_e1_bp1,3,-50,50,33


In [13]:
gridss_callset_copy = gridss_callset.copy()
gridss_meta_copy = gridss_meta.copy()

gridss_callset_cipos = gridss_callset_copy['INFO_CIPOS'].str.split(',', expand=True).fillna('0').astype(int)
gridss_callset_ciposl, gridss_callset_ciposr = gridss_callset_cipos.loc[:, 0], gridss_callset_cipos.loc[:, 1]

gridss_callset_copy, gridss_meta_copy = sv_utils.add_vcf_col(gridss_callset_copy, gridss_meta_copy, 'INFO_CIPOS_l', gridss_callset_ciposl, description='Left CIPOS')
gridss_callset_copy, gridss_meta_copy = sv_utils.add_vcf_col(gridss_callset_copy, gridss_meta_copy, 'INFO_CIPOS_r', gridss_callset_ciposr, description='Right CIPOS')

sv_utils.delete_mult_val_cols(gridss_callset_copy, gridss_meta_copy, vocal=True)
sv_utils.convert_col_types(gridss_callset_copy, gridss_meta_copy)

Dropping entries ['FILTER_ASSEMBLY_BIAS', 'FILTER_ASSEMBLY_ONLY', 'FILTER_ASSEMBLY_TOO_FEW_READ', 'FILTER_ASSEMBLY_TOO_SHORT', 'FILTER_INSUFFICIENT_SUPPORT', 'FILTER_LOW_MAPQ', 'FILTER_LOW_QUAL', 'FILTER_NO_ASSEMBLY', 'FILTER_NO_RP', 'FILTER_NO_SR', 'FILTER_REF', 'FILTER_SINGLE_ASSEMBLY', 'FILTER_SINGLE_SUPPORT', 'FILTER_SMALL_EVENT', 'FILTER_PASS', 'INFO_BEALN', 'INFO_BEID', 'INFO_BEIDH', 'INFO_BEIDL', 'INFO_BENAMES', 'INFO_BPNAMES', 'INFO_CIPOS', 'INFO_CIRPOS', 'INFO_HOMLEN', 'INFO_HOMSEQ', 'INFO_IHOMPOS', 'INFO_IMPRECISE', 'INFO_INSRM', 'INFO_MATEID', 'INFO_RSI', 'INFO_SELF', 'INFO_SI', 'FORMAT_AF'] since they contain multiple values


In [14]:
gridss_callset_copy = gridss_callset_copy.reset_index(drop=True)

In [15]:
gridss_bnd_alts = sv_utils.encode_bnd_alts(gridss_callset_copy['ALT'])
gridss_callset_copy.loc[:, 'bnd_pos_encoding'] = gridss_bnd_alts

/vast/ai_projects/Papenfuss_SVEnsemble2/model/svensemble/utils.py:295: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  encodings = encoding_df.idxmax(axis=1)


In [16]:
gridss_callset_bps = bndc.bnd_to_bp(gridss_callset_copy, force_diff_cols=['bnd_pos_encoding'])

gridss_bptype = gridss_callset_bps['bnd_pos_encoding_bnd1'].astype(str) + gridss_callset_bps['bnd_pos_encoding_bnd2'].astype(str)
gridss_callset_bps.loc[:, 'bptype'] = gridss_bptype

{'chr1': 0, 'chr2': 1, 'chr3': 2, 'chr4': 3, 'chr5': 4, 'chr6': 5, 'chr7': 6, 'chr8': 7, 'chr9': 8, 'chr10': 9, 'chr11': 10, 'chr12': 11, 'chr13': 12, 'chr14': 13, 'chr15': 14, 'chr16': 15, 'chr17': 16, 'chr18': 17, 'chr19': 18, 'chr20': 19, 'chr21': 20, 'chr22': 21, 'chrX': 22, 'chrY': 23, 'chrM': 24}


In [17]:
def get_bnd_prs(bp_vcf_df, padding='INFO_CIPOS', suffixes=None, retain_cols=None):
    """Extract breakend coordinates from a breakpoint VCF dataframe.
    
    Takes a VCF dataframe in breakpoint notation and extracts the coordinates for both 
    breakends, returning them as PyRanges objects. Can handle different padding fields
    and retain additional columns from the input.

    Args:
        bp_vcf_df (pd.DataFrame): DataFrame containing breakpoint VCF data
        padding (str or tuple): Either:
            - Name of padding field in VCF (e.g. 'INFO_CIPOS') to use for coordinate calculation
            - Tuple of (left_padding, right_padding) integers specifying how many bases to extend 
              from each breakpoint position in both directions
            Defaults to 'INFO_CIPOS'.
        suffixes (tuple, optional): Tuple of suffixes used to identify first and second breakends.
            Defaults to ('_bnd1', '_bnd2').
        retain_cols (list, optional): Additional columns to retain from input DataFrame.
            Defaults to None.

    Returns:
        tuple: (bnd1_coords, bnd2_coords) - Two PyRanges objects containing coordinates
            for first and second breakends respectively.

    Raises:
        ValueError: If padding argument is not a string or tuple of integers.
    """
    if suffixes is None:
        suffixes = ('_bnd1', '_bnd2')
    
    if isinstance(padding, tuple):
        if not all(isinstance(x, int) for x in padding):
            raise ValueError('When padding is a tuple, it must contain two integers')
        padding_cols_bnd1 = padding
        padding_cols_bnd2 = padding
    elif isinstance(padding, str):
        padding_cols_bnd1 = tuple([bp_vcf_df[f'{padding}_{side}_{suffixes[0]}'] for side in ('l', 'r')])
        padding_cols_bnd2 = tuple([bp_vcf_df[f'{padding}_{side}_{suffixes[1]}'] for side in ('l', 'r')])
    else:
        raise ValueError('padding must be a string or tuple of integers')
    
    if retain_cols is None:
        retain_cols = []

    bnd1_coords = pd.DataFrame()
    bnd2_coords = pd.DataFrame()

    bnd1_suffix = suffixes[0] 
    bnd2_suffix = suffixes[1]

    # Extract first breakend coordinates
    bnd1_coords.loc[:, 'Chromosome'] = bp_vcf_df[f'CHROM{bnd1_suffix}']
    bnd1_coords.loc[:, 'Start'] = (bp_vcf_df[f'POS{bnd1_suffix}'] - 1 + 
                                  padding_cols_bnd1[0])
    bnd1_coords.loc[:, 'End'] = (bp_vcf_df[f'POS{bnd1_suffix}'] + 
                                padding_cols_bnd1[1])
    bnd1_coords.loc[:, 'ID'] = bp_vcf_df['INFO_EVENT']
    for col in retain_cols:
        if not col.endswith(bnd2_suffix):
            bnd1_coords.loc[:, col] = bp_vcf_df[col]

    # Extract second breakend coordinates  
    bnd2_coords.loc[:, 'Chromosome'] = bp_vcf_df[f'CHROM{bnd2_suffix}']
    bnd2_coords.loc[:, 'Start'] = (bp_vcf_df[f'POS{bnd2_suffix}'] - 1 + 
                                  padding_cols_bnd2[0])
    bnd2_coords.loc[:, 'End'] = (bp_vcf_df[f'POS{bnd2_suffix}'] + 
                                padding_cols_bnd2[1])
    bnd2_coords.loc[:, 'ID'] = bp_vcf_df['INFO_EVENT']
    for col in retain_cols:
        if not col.endswith(bnd1_suffix):
            bnd2_coords.loc[:, col] = bp_vcf_df[col]

    return pr.PyRanges(bnd1_coords), pr.PyRanges(bnd2_coords)



def find_matching_breakpoint_ids(vcf_df1, vcf_df2):
    """
    Find IDs of matching breakpoints between two VCF dataframes.
    
    Args:
        vcf_df1, vcf_df2: VCF dataframes in breakpoint notation with CIPOS fields
    Returns:
        Array of IDs from vcf_df1 that have matching breakpoints in vcf_df2
    """
    # Get pyranges objects for both breakends of each VCF
    bnd1_pr1, bnd2_pr1 = get_bnd_prs(vcf_df1, retain_cols=['bptype'])
    bnd1_pr2, bnd2_pr2 = get_bnd_prs(vcf_df2, retain_cols=['bptype'])

    # Find overlapping regions between corresponding breakends
    joined_bnd1 = bnd1_pr1.join(bnd1_pr2, how=None, report_overlap=True).df
    joined_bnd2 = bnd2_pr1.join(bnd2_pr2, how=None, report_overlap=True).df
    
    # Calculate overlap fraction relative to both interval lengths
    joined_bnd1.loc[:, 'overlap_frac'] = np.maximum(
        joined_bnd1['Overlap'] / (joined_bnd1['End'] - joined_bnd1['Start']),
        joined_bnd1['Overlap'] / (joined_bnd1['End_b'] - joined_bnd1['Start_b'])
    )
    joined_bnd2.loc[:, 'overlap_frac'] = np.maximum(
        joined_bnd2['Overlap'] / (joined_bnd2['End'] - joined_bnd2['Start']),
        joined_bnd2['Overlap'] / (joined_bnd2['End_b'] - joined_bnd2['Start_b'])
    )

    # Get IDs with >80% overlap and matching bptypes
    bnd1_ids = joined_bnd1[
        (joined_bnd1['overlap_frac'] > 0.8) &\
        (joined_bnd1['bptype'] == joined_bnd1['bptype_b'])
    ]['ID'].unique()
    bnd2_ids = joined_bnd2[
        (joined_bnd2['overlap_frac'] > 0.8) &\
        (joined_bnd2['bptype'] == joined_bnd2['bptype_b'])
    ]['ID'].unique()

    # Return IDs matching in both breakends
    return bnd1_ids[np.isin(bnd1_ids, bnd2_ids)]


In [18]:
cipos_cols = ['INFO_CIPOS_l_bnd1', 'INFO_CIPOS_r_bnd1', 'INFO_CIPOS_l_bnd2', 'INFO_CIPOS_r_bnd2']

# Get breakend coordinates for GRIDSS callset
gridss_bnd1_pr, gridss_bnd2_pr = get_bnd_prs(gridss_callset_bps, padding=(0, 0), retain_cols=['bptype'] + cipos_cols)

# Get breakend coordinates for reference callset
ref_bnd1_pr, ref_bnd2_pr = get_bnd_prs(ref_callset_bps, padding=(0, 0), retain_cols=['bptype'] + cipos_cols)


In [19]:
gridss_callset_bps = bndc.sort_bnd_df(gridss_callset_bps, chrom_keys=sv_utils.generate_chrom_key_dict(pd.concat([gridss_callset_bps['CHROM_bnd1'], gridss_callset_bps['CHROM_bnd2']])))

In [20]:
gridss_callset_bps

,CHROM_bnd1,POS_bnd1,ID_bnd1,REF_bnd1,ALT_bnd1,QUAL_bnd1,FORMAT_ANRP_bnd1,FORMAT_ANRPQ_bnd1,FORMAT_ANSR_bnd1,FORMAT_ANSRQ_bnd1,FORMAT_ASQ_bnd1,FORMAT_ASRP_bnd1,FORMAT_ASSR,FORMAT_BAQ,FORMAT_BASRP,FORMAT_BASSR,FORMAT_BQ_bnd1,FORMAT_BSC_bnd1,FORMAT_BSCQ_bnd1,FORMAT_BUM_bnd1,FORMAT_BUMQ_bnd1,FORMAT_BVF_bnd1,FORMAT_CASQ_bnd1,FORMAT_GT,FORMAT_IC,FORMAT_IQ,FORMAT_QUAL_bnd1,FORMAT_RASQ_bnd1,FORMAT_REF_bnd1,FORMAT_REFPAIR_bnd1,FORMAT_RP,FORMAT_RPQ,FORMAT_SR,FORMAT_SRQ,FORMAT_VF_bnd1,INFO_SVLEN,INFO_ANRP_bnd1,INFO_ANRPQ_bnd1,INFO_ANSR_bnd1,INFO_ANSRQ_bnd1,INFO_AS_bnd1,INFO_ASC_bnd1,INFO_ASQ_bnd1,INFO_ASRP_bnd1,INFO_ASSR,INFO_BA,INFO_BAQ,INFO_BASRP,INFO_BASSR,INFO_BMQ_bnd1,INFO_BMQN_bnd1,INFO_BMQX_bnd1,INFO_BQ_bnd1,INFO_BSC_bnd1,INFO_BSCQ_bnd1,INFO_BUM_bnd1,INFO_BUMQ_bnd1,INFO_BVF_bnd1,INFO_CAS_bnd1,INFO_CASQ_bnd1,INFO_CQ,INFO_EVENT,INFO_IC,INFO_INSRMP,INFO_INSRMRC,INFO_INSRMRO,INFO_INSRMRT,INFO_INSTAXID,INFO_IQ,INFO_MQ_bnd1,INFO_MQN,INFO_MQX,INFO_RAS_bnd1,INFO_RASQ_bnd1,INFO_REF_bnd1,INFO_REFPAIR_bnd1,INFO_RF,INFO_RP,INFO_RPQ,INFO_SB_bnd1,INFO_SC_bnd1,INFO_SR,INFO_SRQ,INFO_SVTYPE,INFO_VF_bnd1,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,bnd_pos_encoding_bnd1,CHROM_bnd2,POS_bnd2,ID_bnd2,REF_bnd2,ALT_bnd2,QUAL_bnd2,FORMAT_ANRP_bnd2,FORMAT_ANRPQ_bnd2,FORMAT_ANSR_bnd2,FORMAT_ANSRQ_bnd2,FORMAT_ASQ_bnd2,FORMAT_ASRP_bnd2,FORMAT_BQ_bnd2,FORMAT_BSC_bnd2,FORMAT_BSCQ_bnd2,FORMAT_BUM_bnd2,FORMAT_BUMQ_bnd2,FORMAT_BVF_bnd2,FORMAT_CASQ_bnd2,FORMAT_QUAL_bnd2,FORMAT_RASQ_bnd2,FORMAT_REF_bnd2,FORMAT_REFPAIR_bnd2,FORMAT_VF_bnd2,INFO_ANRP_bnd2,INFO_ANRPQ_bnd2,INFO_ANSR_bnd2,INFO_ANSRQ_bnd2,INFO_AS_bnd2,INFO_ASC_bnd2,INFO_ASQ_bnd2,INFO_ASRP_bnd2,INFO_BMQ_bnd2,INFO_BMQN_bnd2,INFO_BMQX_bnd2,INFO_BQ_bnd2,INFO_BSC_bnd2,INFO_BSCQ_bnd2,INFO_BUM_bnd2,INFO_BUMQ_bnd2,INFO_BVF_bnd2,INFO_CAS_bnd2,INFO_CASQ_bnd2,INFO_MQ_bnd2,INFO_RAS_bnd2,INFO_RASQ_bnd2,INFO_REF_bnd2,INFO_REFPAIR_bnd2,INFO_SB_bnd2,INFO_SC_bnd2,INFO_VF_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,bnd_pos_encoding_bnd2,bptype
29,chr1,9952,gridss0bf_1o,N,]chr1:10498]N,498.42,27,408.2,0,0.00,0.00,0,0,0.0,0,0,829.69,0,0.00,38,829.69,38,0.0,.,0,0.00,498.42,0.00,0,0,33,498.42,0,0.00,33,0,27,408.2,0,0.00,0,1X96N1X,0.00,0,0,0,0.0,0,0,29.16,20.0,59.0,829.69,0,0.00,38,829.69,38,0,0.0,498.42,gridss0bf_1,0,0.0,0,0,0,0,0.00,33.03,20.0,59.0,0,0.00,0,0,0,33,498.42,0.000000,1X96N1X139M36D199M,0,0.00,BND,33,-48,49,2,chr1,10498,gridss0bf_1h,G,G[chr1:9952[,498.42,28,422.92,0,0.00,0.00,0,655.32,8,133.81,24,521.51,31,0.0,498.42,0.00,43,12,33,28,422.92,0,0.00,0,1X57N1X,0.00,0,29.56,20.0,56.0,655.32,8,133.81,24,521.51,31,0,0.0,33.03,0,0.00,43,12,0.500000,95M34D305M1X57N1X,33,-29,29,1,21
165,chr1,10317,gridss0fb_13o,A,ACCCCTTCCCC[chr1:10328[,241.38,0,0.0,0,0.00,0.00,0,2,0.0,0,0,35.40,2,35.40,0,0.00,2,0.0,.,7,194.16,241.38,47.22,443,500,0,0.00,0,0.00,9,0,0,0.0,0,0.00,0,1X,0.00,0,2,0,0.0,0,0,41.50,38.0,45.0,35.40,2,35.40,0,0.00,2,0,0.0,194.16,gridss0fb_13,7,0.0,0,0,0,0,194.16,41.13,29.0,49.0,1,47.22,443,500,0,0,0.00,0.000000,11M1D68M1X,0,0.00,BND,9,0,0,1,chr1,10328,gridss0fb_13h,A,]chr1:10317]CCCCTTCCCCA,241.38,0,0.00,0,0.00,47.22,0,0.00,0,0.00,0,0.00,0,0.0,241.38,0.00,450,482,9,0,0.00,0,0.00,1,1X2N1X,47.22,0,0.00,0.0,0.0,0.00,0,0.00,0,0.00,0,0,0.0,41.13,0,0.00,450,482,0.000000,1X2N1X56M,9,0,0,2,12
169,chr1,10325,gridss0fb_14o,C,CCAACCCC[chr1:10333[,230.11,0,0.0,0,0.00,0.00,0,5,0.0,0,0,76.37,4,76.37,0,0.00,4,0.0,.,4,101.09,230.11,129.02,454,481,0,0.00,0,0.00,4,0,0,0.0,0,0.00,0,1X,0.00,0,5,0,0.0,0,0,40.75,36.0,47.0,76.37,4,76.37,0,0.00,4,0,0.0,200.56,gridss0fb_14,4,0.0,0,0,0,0,101.09,31.20,20.0,49.0,1,129.02,454,481,0,0,0.00,0.263158,55M1X,0,0.00,BND,4,0,0,1,chr1,10333,gridss0fb_14h,C,]chr1:10325]CAACCCCC,230.11,0,0.00,0,0.00,129.02,0,111.57,7,111.57,0,0.00,7,0.0,230.11,0.00,487,452,4,0,0.00,0,0.00,1,1X,129.02,0,28.71,23.0,40.0,111.57,7,111.57,0,0.00,7,0,0.0,31.20,0,0.00,487,452,0.136364,1X75M,4,0,0,2,12
301,chr1,10377,gridss0fb_27o,A,AACCCTTAACCCTAACCCTAACCCCTAACCCCTAACCCTAACCCTA...,105.80,0,0.0,0,0.00,105.80,7,0,0.0,0,0,22.15,0,0.00,1,22.1

In [112]:
gridss_callset_bps.loc[:, 'chrom_bptype'] = gridss_callset_bps['CHROM_bnd1'] + '_' + gridss_callset_bps['CHROM_bnd2'] + '_' + gridss_callset_bps['bptype']
ref_callset_bps.loc[:, 'chrom_bptype'] = ref_callset_bps['CHROM_bnd1'] + '_' + ref_callset_bps['CHROM_bnd2'] + '_' + ref_callset_bps['bptype']

gridss_unique_chrom_bptypes = gridss_callset_bps['chrom_bptype'].unique()

In [114]:
len(gridss_unique_chrom_bptypes)


1157

In [115]:
kdtforest = {
    chrom_bptype: cKDTree(ref_callset_bps[ref_callset_bps['chrom_bptype'] == chrom_bptype][['POS_bnd1', 'POS_bnd2']].to_numpy())
    for chrom_bptype in gridss_unique_chrom_bptypes
}

In [116]:
ref_closes_to_gridss = {
    chrom_bptype: kdtforest[chrom_bptype].query(gridss_callset_bps[gridss_callset_bps['chrom_bptype'] == chrom_bptype][['POS_bnd1', 'POS_bnd2']].to_numpy())
    for chrom_bptype in gridss_unique_chrom_bptypes
}


In [118]:
ref_closes_to_gridss['chr1_chr1_21'][1]

array([    0,     8,    70, 10553,    74,    76,    78,   132,   159,
         189,   261,   296,   366,   376,   390,   420,   432,   450,
         455,   466,   484,   549,   580,   587,   587,   589,   612,
         668,   694,   728,   875,   875,   736,   759,   875,   768,
         770,   780,   791,   893,   908,   918,   918,  1011,  1075,
        1114,  1118,  1138,  1157,  1172,  1183,  1197,  1208,  1218,
        1221,  1236,  1260,  1267,  1306,  1312,  1319,  1342,  1354,
        1397,  1418,  1446,  1446,  1539,  1539,  1555,  1559,  1559,
        1614,  1621,  1621,  1631,  1642,  1651,  1655,  1667,  1697,
        1697,  1697,  1722,  1749,  2165,  1771,  1775,  1814,  1882,
        1954,  1959,  1974,  2098,  2110,  2169,  2190,  2195,  2199,
        2236,  2237,  2386,  2425,  2425,  2436,  2446,  2478,  2477,
        2665,  2667,  2741,  2814,  2828,  2837,  2840,  2898,  2925,
        2942,  2987,  2998,  2998,  3032,  3033,  3033,  3033,  3115,
        3117,  5675,

In [132]:
ref_matches = pd.DataFrame()

for chrom_bptype in gridss_unique_chrom_bptypes:
    # Filter reference breakpoints by chromosomes and breakpoint type
    filtered_refs = ref_callset_bps[
        ref_callset_bps['chrom_bptype'] == chrom_bptype
    ]
    
    if filtered_refs.shape[0] == 0:
        continue
        
    # Reset index to ensure proper indexing when selecting matches
    filtered_refs = filtered_refs.reset_index(drop=True)
    
    # Get indices of closest reference breakpoints
    matched_indices = ref_closes_to_gridss[chrom_bptype][1]
    
    # Select matching reference breakpoints and add to results
    matched_refs = filtered_refs.iloc[matched_indices]
    ref_matches = pd.concat([ref_matches, matched_refs])

ref_matches

,CHROM_bnd1,POS_bnd1,ID_bnd1,REF,ALT_bnd1,QUAL_bnd1,INFO_CHR2,INFO_END,INFO_SVLEN,INFO_SVTYPE,INFO_EVIDENCE_SR,INFO_EVIDENCE_PE,INFO_EVIDENCE_RD,INFO_EVIDENCE_BEST,INFO_EVENT,INFO_MATEID_bnd1,bnd_pos_encoding_bnd1,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,CHROM_bnd2,POS_bnd2,ID_bnd2,ALT_bnd2,QUAL_bnd2,INFO_MATEID_bnd2,bnd_pos_encoding_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,bptype,chrom_comb,chrom_bptype
0,chr1,11000,gnomAD-SV_v3_DUP_chr1_7d73682f_bp2,N,]chr1:51000]N,999.0,chr1,51000,40000,DUP,0,0,1,RD,gnomAD-SV_v3_DUP_chr1_7d73682f,gnomAD-SV_v3_DUP_chr1_7d73682f_bp1,2,-500,500,chr1,51000,gnomAD-SV_v3_DUP_chr1_7d73682f_bp1,N[chr1:51000[,999.0,gnomAD-SV_v3_DUP_chr1_7d73682f_bp2,1,-500,500,21,chr1_chr1,chr1_chr1_21
8,chr1,180867,gnomAD-SV_v3_DUP_chr1_4c029562_bp2,N,]chr1:180942]N,34.0,chr1,180942,75,DUP,1,0,1,SR,gnomAD-SV_v3_DUP_chr1_4c029562,gnomAD-SV_v3_DUP_chr1_4c029562_bp1,2,-50,50,chr1,180942,gnomAD-SV_v3_DUP_chr1_4c029562_bp1,N[chr1:180942[,34.0,gnomAD-SV_v3_DUP_chr1_4c029562_bp2,1,-50,50,21,chr1_chr1,chr1_chr1_21
70,chr1,789472,gnomAD-SV_v3_DUP_chr1_7a62ac0e_bp2,N,]chr1:790122]N,1.0,chr1,790122,650,DUP,1,0,0,SR,gnomAD-SV_v3_DUP_chr1_7a62ac0e,gnomAD-SV_v3_DUP_chr1_7a62ac0e_bp1,2,-50,50,chr1,790122,gnomAD-SV_v3_DUP_chr1_7a62ac0e_bp1,N[chr1:790122[,1.0,gnomAD-SV_v3_DUP_chr1_7a62ac0e_bp2,1,-50,50,21,chr1_chr1,chr1_chr1_21
10553,chr1,114949917,gnomAD-SV_v3_DUP_chr1_c2c08a23_bp2,N,]chr1:121518267]N,147.0,chr1,121518267,6568350,DUP,0,0,1,RD,gnomAD-SV_v3_DUP_chr1_c2c08a23,gnomAD-SV_v3_DUP_chr1_c2c08a23_bp1,2,-500,500,chr1,121518267,gnomAD-SV_v3_DUP_chr1_c2c08a23_bp1,N[chr1:121518267[,147.0,gnomAD-SV_v3_DUP_chr1_c2c08a23_bp2,1,-500,500,21,chr1_chr1,chr1_chr1_21
74,chr1,789869,gnomAD-SV_v3_DUP_chr1_344b81a9_bp2,N,]chr1:790168]N,1.0,chr1,790168,299,DUP,1,1,0,SR,gnomAD-SV_v3_DUP_chr1_344b81a9,gnomAD-SV_v3_DUP_chr1_344b81a9_bp1,2,-50,50,chr1,790168,gnomAD-SV_v3_DUP_chr1_344b81a9_bp1,N[chr1:790168[,1.0,gnomAD-SV_v3_DUP_chr1_344b81a9_bp2,1,-50,50,21,chr1_chr1,chr1_chr1_21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,chrX,145641088,gnomAD-SV_v3_INV_chrX_73ab2c16_bp1,N,N]chrX:152809890],465.0,chrX,152809890,7168801,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_73ab2c16_e1,gnomAD-SV_v3_INV_chrX_73ab2c16_e1_bp3,3,-50,50,chrX,152809890,gnomAD-SV_v3_INV_chrX_73ab2c16_bp3,N]chrX:152809889],465.0,gnomAD-SV_v3_INV_chrX_73ab2c16_e1_bp1,3,-50,50,33,chrX_chrX,chrX_chrX_33
53,chrX,136450316,gnomAD-SV_v3_INV_chrX_caa41ad4_bp1,N,N]chrX:138402970],976.0,chrX,138402970,1952653,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_caa41ad4_e1,gnomAD-SV_v3_INV_chrX_caa41ad4_e1_bp3,3,-50,50,chrX,138402970,gnomAD-SV_v3_INV_chrX_caa41ad4_bp3,N]chrX:138402969],976.0,gnomAD-SV_v3_INV_chrX_caa41ad4_e1_bp1,3,-50,50,33,chrX_chrX,chrX_chrX_33
53,chrX,136450316,gnomAD-SV_v3_INV_chrX_caa41ad4_bp1,N,N]chrX:138402970],976.0,chrX,138402970,1952653,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_caa41ad4_e1,gnomAD-SV_v3_INV_chrX_caa41ad4_e1_bp3,3,-50,50,chrX,138402970,gnomAD-SV_v3_INV_chrX_caa41ad4_bp3,N]chrX:138402969],976.0,gnomAD-SV_v3_INV_chrX_caa41ad4_e1_bp1,3,-50,50,33,chrX_chrX,chrX_chrX_33
57,chrX,146621846,gnomAD-SV_v3_INV_chrX_2fc8f462_bp1,N,N]chrX:147549974],815.0,chrX,147549974,928127,INV,1,1,0,SR,gnomAD-SV_v3_INV_chrX_2fc8f462_e1,gnomAD-SV_v3_INV_chrX_2fc8f462_e1_bp3,3,-50,50,chrX,147549974,gnomAD-SV_v3_INV_chrX_2fc8f462_bp3,N]chrX:147549973],815.0,gnomAD-SV_v3_INV_chrX_2fc8f462_e1_bp1,3,-50,50,33,chrX_chrX,chrX_chrX_33


In [135]:
gridss_callset_bps[gridss_callset_bps['chrom_bptype'].isin(ref_matches['chrom_bptype'].unique())]


,CHROM_bnd1,POS_bnd1,ID_bnd1,REF_bnd1,ALT_bnd1,QUAL_bnd1,FORMAT_ANRP_bnd1,FORMAT_ANRPQ_bnd1,FORMAT_ANSR_bnd1,FORMAT_ANSRQ_bnd1,FORMAT_ASQ_bnd1,FORMAT_ASRP_bnd1,FORMAT_ASSR,FORMAT_BAQ,FORMAT_BASRP,FORMAT_BASSR,FORMAT_BQ_bnd1,FORMAT_BSC_bnd1,FORMAT_BSCQ_bnd1,FORMAT_BUM_bnd1,FORMAT_BUMQ_bnd1,FORMAT_BVF_bnd1,FORMAT_CASQ_bnd1,FORMAT_GT,FORMAT_IC,FORMAT_IQ,FORMAT_QUAL_bnd1,FORMAT_RASQ_bnd1,FORMAT_REF_bnd1,FORMAT_REFPAIR_bnd1,FORMAT_RP,FORMAT_RPQ,FORMAT_SR,FORMAT_SRQ,FORMAT_VF_bnd1,INFO_SVLEN,INFO_ANRP_bnd1,INFO_ANRPQ_bnd1,INFO_ANSR_bnd1,INFO_ANSRQ_bnd1,INFO_AS_bnd1,INFO_ASC_bnd1,INFO_ASQ_bnd1,INFO_ASRP_bnd1,INFO_ASSR,INFO_BA,INFO_BAQ,INFO_BASRP,INFO_BASSR,INFO_BMQ_bnd1,INFO_BMQN_bnd1,INFO_BMQX_bnd1,INFO_BQ_bnd1,INFO_BSC_bnd1,INFO_BSCQ_bnd1,INFO_BUM_bnd1,INFO_BUMQ_bnd1,INFO_BVF_bnd1,INFO_CAS_bnd1,INFO_CASQ_bnd1,INFO_CQ,INFO_EVENT,INFO_IC,INFO_INSRMP,INFO_INSRMRC,INFO_INSRMRO,INFO_INSRMRT,INFO_INSTAXID,INFO_IQ,INFO_MQ_bnd1,INFO_MQN,INFO_MQX,INFO_RAS_bnd1,INFO_RASQ_bnd1,INFO_REF_bnd1,INFO_REFPAIR_bnd1,INFO_RF,INFO_RP,INFO_RPQ,INFO_SB_bnd1,INFO_SC_bnd1,INFO_SR,INFO_SRQ,INFO_SVTYPE,INFO_VF_bnd1,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,bnd_pos_encoding_bnd1,CHROM_bnd2,POS_bnd2,ID_bnd2,REF_bnd2,ALT_bnd2,QUAL_bnd2,FORMAT_ANRP_bnd2,FORMAT_ANRPQ_bnd2,FORMAT_ANSR_bnd2,FORMAT_ANSRQ_bnd2,FORMAT_ASQ_bnd2,FORMAT_ASRP_bnd2,FORMAT_BQ_bnd2,FORMAT_BSC_bnd2,FORMAT_BSCQ_bnd2,FORMAT_BUM_bnd2,FORMAT_BUMQ_bnd2,FORMAT_BVF_bnd2,FORMAT_CASQ_bnd2,FORMAT_QUAL_bnd2,FORMAT_RASQ_bnd2,FORMAT_REF_bnd2,FORMAT_REFPAIR_bnd2,FORMAT_VF_bnd2,INFO_ANRP_bnd2,INFO_ANRPQ_bnd2,INFO_ANSR_bnd2,INFO_ANSRQ_bnd2,INFO_AS_bnd2,INFO_ASC_bnd2,INFO_ASQ_bnd2,INFO_ASRP_bnd2,INFO_BMQ_bnd2,INFO_BMQN_bnd2,INFO_BMQX_bnd2,INFO_BQ_bnd2,INFO_BSC_bnd2,INFO_BSCQ_bnd2,INFO_BUM_bnd2,INFO_BUMQ_bnd2,INFO_BVF_bnd2,INFO_CAS_bnd2,INFO_CASQ_bnd2,INFO_MQ_bnd2,INFO_RAS_bnd2,INFO_RASQ_bnd2,INFO_REF_bnd2,INFO_REFPAIR_bnd2,INFO_SB_bnd2,INFO_SC_bnd2,INFO_VF_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,bnd_pos_encoding_bnd2,bptype,chrom_comb,chrom_bptype
29,chr1,9952,gridss0bf_1o,N,]chr1:10498]N,498.42,27,408.2,0,0.0,0.0,0,0,0.0,0,0,829.69,0,0.00,38,829.69,38,0.00,.,0,0.00,498.42,0.00,0,0,33,498.42,0,0.0,33,0,27,408.2,0,0.0,0,1X96N1X,0.0,0,0,0,0.0,0,0,29.16,20.0,59.0,829.69,0,0.00,38,829.69,38,0,0.00,498.42,gridss0bf_1,0,0.0,0,0,0,0,0.00,33.03,20.0,59.0,0,0.00,0,0,0,33,498.42,0.000000,1X96N1X139M36D199M,0,0.0,BND,33,-48,49,2,chr1,10498,gridss0bf_1h,G,G[chr1:9952[,498.42,28,422.92,0,0.0,0.00,0,655.32,8,133.81,24,521.51,31,0.00,498.42,0.0,43,12,33,28,422.92,0,0.0,0,1X57N1X,0.00,0,29.56,20.0,56.0,655.32,8,133.81,24,521.51,31,0,0.00,33.03,0,0.0,43,12,0.500000,95M34D305M1X57N1X,33,-29,29,1,21,chr1_chr1,chr1_chr1_21
165,chr1,10317,gridss0fb_13o,A,ACCCCTTCCCC[chr1:10328[,241.38,0,0.0,0,0.0,0.0,0,2,0.0,0,0,35.40,2,35.40,0,0.00,2,0.00,.,7,194.16,241.38,47.22,443,500,0,0.00,0,0.0,9,0,0,0.0,0,0.0,0,1X,0.0,0,2,0,0.0,0,0,41.50,38.0,45.0,35.40,2,35.40,0,0.00,2,0,0.00,194.16,gridss0fb_13,7,0.0,0,0,0,0,194.16,41.13,29.0,49.0,1,47.22,443,500,0,0,0.00,0.000000,11M1D68M1X,0,0.0,BND,9,0,0,1,chr1,10328,gridss0fb_13h,A,]chr1:10317]CCCCTTCCCCA,241.38,0,0.00,0,0.0,47.22,0,0.00,0,0.00,0,0.00,0,0.00,241.38,0.0,450,482,9,0,0.00,0,0.0,1,1X2N1X,47.22,0,0.00,0.0,0.0,0.00,0,0.00,0,0.00,0,0,0.00,41.13,0,0.0,450,482,0.000000,1X2N1X56M,9,0,0,2,12,chr1_chr1,chr1_chr1_12
169,chr1,10325,gridss0fb_14o,C,CCAACCCC[chr1:10333[,230.11,0,0.0,0,0.0,0.0,0,5,0.0,0,0,76.37,4,76.37,0,0.00,4,0.00,.,4,101.09,230.11,129.02,454,481,0,0.00,0,0.0,4,0,0,0.0,0,0.0,0,1X,0.0,0,5,0,0.0,0,0,40.75,36.0,47.0,76.37,4,76.37,0,0.00,4,0,0.00,200.56,gridss0fb_14,4,0.0,0,0,0,0,101.09,31.20,20.0,49.0,1,129.02,454,481,0,0,0.00,0.263158,55M1X,0,0.0,BND,4,0,0,1,chr1,10333,gridss0fb_14h,C,]chr1:10325]CAACCCCC,230.11,0,0.00,0,0.0,129.02,0,111.57,7,111.57,0,0.00,7,0.00,230.11,0.0,487,452,4,0,0.00,0,0.0,1,1X,129.02,0,28.71,23.0,40.0,111.57,7,111.57,0,0.00,7,0,0.00,31.20,0,0.0,487,452,0.136364,1X75M,4,0,0,2,12,chr1_chr1,chr1_chr1_12
301,chr1,10377,gridss0fb_27o,A,AACCCTTAACCCTAACCCTAACCCCTAACCCCT

In [156]:
gridss_ref_closest = pd.concat([gridss_callset_bps[gridss_callset_bps['chrom_bptype'].isin(ref_matches['chrom_bptype'].unique())].reset_index(drop=True), ref_matches.reset_index(drop=True).add_suffix('_ref')], axis=1)
gridss_ref_closest.loc[:, 'eucl_dist'] = np.sqrt((gridss_ref_closest['POS_bnd1'] - gridss_ref_closest['POS_bnd1_ref'])**2 + (gridss_ref_closest['POS_bnd2'] - gridss_ref_closest['POS_bnd2_ref'])**2)
gridss_ref_closest.loc[:, 'manhattan_dist'] = np.abs(gridss_ref_closest['POS_bnd1'] - gridss_ref_closest['POS_bnd1_ref']) + np.abs(gridss_ref_closest['POS_bnd2'] - gridss_ref_closest['POS_bnd2_ref'])


# gridss_ref_closest.loc[:, 'dist_cutoff'] = np.sqrt(
#     (
#         (-gridss_ref_closest['INFO_CIPOS_l_bnd1'] + gridss_ref_closest['INFO_CIPOS_r_bnd1']) / 2 
#         + 100000 
#         - gridss_ref_closest['INFO_CIPOS_l_bnd1_ref']
#     ) ** 2 
#     + (
#         (gridss_ref_closest['INFO_CIPOS_l_bnd2'] + gridss_ref_closest['INFO_CIPOS_r_bnd2']) / 2
#         + 100000 
#         - gridss_ref_closest['INFO_CIPOS_l_bnd2_ref']
#     ) ** 2
# )

gridss_ref_closest.loc[:, 'is_close'] = gridss_ref_closest['manhattan_dist'] < 1000
gridss_ref_closest_test = gridss_ref_closest[gridss_ref_closest['is_close']][['CHROM_bnd1', 'POS_bnd1', 'CHROM_bnd2', 'POS_bnd2', 'CHROM_bnd1_ref', 'POS_bnd1_ref', 'CHROM_bnd2_ref', 'POS_bnd2_ref', 'manhattan_dist', 'INFO_EVENT', 'ALT_bnd1', 'INFO_EVENT_ref']]
gridss_ref_closest_test

,CHROM_bnd1,POS_bnd1,CHROM_bnd2,POS_bnd2,CHROM_bnd1_ref,POS_bnd1_ref,CHROM_bnd2_ref,POS_bnd2_ref,manhattan_dist,INFO_EVENT,ALT_bnd1,INFO_EVENT_ref
11881,chr2,9919,chr2,10495,chr2,10255,chr2,10379,452,gridss128bf_865,]chr2:10495]N,gnomAD-SV_v3_DUP_chr2_fe590f97
13848,chr2,34710034,chr2,34710035,chr2,34710021,chr2,34710025,23,gridss132fb_3419,ATATATATATATATGTG[chr2:34710035[,gnomAD-SV_v3_INS_chr2_7e6f1add
24285,chr3,10210,chr3,10752,chr3,10418,chr3,10562,398,gridss169bf_335,]chr3:10752]C,gnomAD-SV_v3_DUP_chr3_6d1b918e
25558,chr3,24778028,chr3,24778354,chr3,24778036,chr3,24778345,17,gridss171fb_6438,T[chr3:24778354[,gnomAD-SV_v3_DEL_chr3_8e9f8dc5
33980,chr4,46841,chr4,47236,chr4,46840,chr4,47372,137,gridss189fb_2371,A[chr4:47236[,gnomAD-SV_v3_DEL_chr4_a37183ee
...,...,...,...,...,...,...,...,...,...,...,...,...
135667,chr22,49973698,chr22,49973711,chr22,49973711,chr22,49973808,110,gridss169fb_2804,G[chr22:49973711[,gnomAD-SV_v3_DUP_chr22_daac2cc1
135669,chr22,49973724,chr22,49973804,chr22,49973711,chr22,49973808,17,gridss169bf_257,]chr22:49973804]G,gnomAD-SV_v3_DUP_chr22_daac2cc1
135672,chr22,49973746,chr22,49973802,chr22,49973711,chr22,49973808,41,gridss169bf_260,]chr22:49973802]TTGTGTGTTTATGGGG,gnomAD-SV_v3_DUP_chr22_daac2cc1
135673,chr22,49973753,chr22,49975685,chr22,49973496,chr22,49975686,258,gridss169bf_261,]chr22:49975685]TGTGCGTGTG,gnomAD-SV_v3_DUP_chr22_f2f333ed


In [128]:
ref_callset_bps[ref_callset_bps['chrom_bptype'] == 'chr2_chr2_21'].iloc[ref_closes_to_gridss['chr2_chr2_21'][1]]

,CHROM_bnd1,POS_bnd1,ID_bnd1,REF,ALT_bnd1,QUAL_bnd1,INFO_CHR2,INFO_END,INFO_SVLEN,INFO_SVTYPE,INFO_EVIDENCE_SR,INFO_EVIDENCE_PE,INFO_EVIDENCE_RD,INFO_EVIDENCE_BEST,INFO_EVENT,INFO_MATEID_bnd1,bnd_pos_encoding_bnd1,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,CHROM_bnd2,POS_bnd2,ID_bnd2,ALT_bnd2,QUAL_bnd2,INFO_MATEID_bnd2,bnd_pos_encoding_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,bptype,chrom_comb,chrom_bptype
775128,chr2,10255,gnomAD-SV_v3_DUP_chr2_fe590f97_bp2,N,]chr2:10379]N,1.0,chr2,10379,124,DUP,1,0,0,SR,gnomAD-SV_v3_DUP_chr2_fe590f97,gnomAD-SV_v3_DUP_chr2_fe590f97_bp1,2,-50,50,chr2,10379,gnomAD-SV_v3_DUP_chr2_fe590f97_bp1,N[chr2:10379[,1.0,gnomAD-SV_v3_DUP_chr2_fe590f97_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
769139,chr2,547219,gnomAD-SV_v3_DUP_chr2_b2a4c666_bp2,N,]chr2:547440]N,1.0,chr2,547440,221,DUP,1,0,0,SR,gnomAD-SV_v3_DUP_chr2_b2a4c666,gnomAD-SV_v3_DUP_chr2_b2a4c666_bp1,2,-50,50,chr2,547440,gnomAD-SV_v3_DUP_chr2_b2a4c666_bp1,N[chr2:547440[,1.0,gnomAD-SV_v3_DUP_chr2_b2a4c666_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
766227,chr2,604847,gnomAD-SV_v3_DUP_chr2_8f1ece4d_bp2,N,]chr2:604933]N,1.0,chr2,604933,86,DUP,1,0,1,SR,gnomAD-SV_v3_DUP_chr2_8f1ece4d,gnomAD-SV_v3_DUP_chr2_8f1ece4d_bp1,2,-50,50,chr2,604933,gnomAD-SV_v3_DUP_chr2_8f1ece4d_bp1,N[chr2:604933[,1.0,gnomAD-SV_v3_DUP_chr2_8f1ece4d_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
757343,chr2,694297,gnomAD-SV_v3_DUP_chr2_1d83bfb1_bp2,N,]chr2:694367]N,718.0,chr2,694367,70,DUP,1,0,1,SR,gnomAD-SV_v3_DUP_chr2_1d83bfb1,gnomAD-SV_v3_DUP_chr2_1d83bfb1_bp1,2,-50,50,chr2,694367,gnomAD-SV_v3_DUP_chr2_1d83bfb1_bp1,N[chr2:694367[,718.0,gnomAD-SV_v3_DUP_chr2_1d83bfb1_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
759989,chr2,740144,gnomAD-SV_v3_DUP_chr2_3fa2824a_bp2,N,]chr2:740746]N,999.0,chr2,740746,602,DUP,1,1,1,SR,gnomAD-SV_v3_DUP_chr2_3fa2824a,gnomAD-SV_v3_DUP_chr2_3fa2824a_bp1,2,-50,50,chr2,740746,gnomAD-SV_v3_DUP_chr2_3fa2824a_bp1,N[chr2:740746[,999.0,gnomAD-SV_v3_DUP_chr2_3fa2824a_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
757886,chr2,241896278,gnomAD-SV_v3_DUP_chr2_2440f4df_bp2,N,]chr2:241896536]N,1.0,chr2,241896536,258,DUP,1,0,0,SR,gnomAD-SV_v3_DUP_chr2_2440f4df,gnomAD-SV_v3_DUP_chr2_2440f4df_bp1,2,-50,50,chr2,241896536,gnomAD-SV_v3_DUP_chr2_2440f4df_bp1,N[chr2:241896536[,1.0,gnomAD-SV_v3_DUP_chr2_2440f4df_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
773238,chr2,241987351,gnomAD-SV_v3_DUP_chr2_e7335ad6_bp2,N,]chr2:241987438]N,1.0,chr2,241987438,87,DUP,1,1,1,SR,gnomAD-SV_v3_DUP_chr2_e7335ad6,gnomAD-SV_v3_DUP_chr2_e7335ad6_bp1,2,-50,50,chr2,241987438,gnomAD-SV_v3_DUP_chr2_e7335ad6_bp1,N[chr2:241987438[,1.0,gnomAD-SV_v3_DUP_chr2_e7335ad6_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
764546,chr2,241989500,gnomAD-SV_v3_DUP_chr2_79d786ee_bp2,N,]chr2:241989773]N,1.0,chr2,241989773,273,DUP,1,1,1,SR,gnomAD-SV_v3_DUP_chr2_79d786ee,gnomAD-SV_v3_DUP_chr2_79d786ee_bp1,2,-50,50,chr2,241989773,gnomAD-SV_v3_DUP_chr2_79d786ee_bp1,N[chr2:241989773[,1.0,gnomAD-SV_v3_DUP_chr2_79d786ee_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21
757536,chr2,242088462,gnomAD-SV_v3_DUP_chr2_1fdaf919_bp2,N,]chr2:242088544]N,57.0,chr2,242088544,82,DUP,1,0,0,SR,gnomAD-SV_v3_DUP_chr2_1fdaf919,gnomAD-SV_v3_DUP_chr2_1fdaf919_bp1,2,-50,50,chr2,242088544,gnomAD-SV_v3_DUP_chr2_1fdaf919_bp1,N[chr2:242088544[,57.0,gnomAD-SV_v3_DUP_chr2_1fdaf919_bp2,1,-50,50,21,chr2_chr2,chr2_chr2_21


In [17]:
gridss_callset_bps

,CHROM_bnd1,POS_bnd1,ID_bnd1,REF_bnd1,ALT_bnd1,QUAL_bnd1,FORMAT_ANRP_bnd1,FORMAT_ANRPQ_bnd1,FORMAT_ANSR_bnd1,FORMAT_ANSRQ_bnd1,FORMAT_ASQ_bnd1,FORMAT_ASRP_bnd1,FORMAT_ASSR,FORMAT_BAQ,FORMAT_BASRP,FORMAT_BASSR,FORMAT_BQ_bnd1,FORMAT_BSC_bnd1,FORMAT_BSCQ_bnd1,FORMAT_BUM_bnd1,FORMAT_BUMQ_bnd1,FORMAT_BVF_bnd1,FORMAT_CASQ_bnd1,FORMAT_GT,FORMAT_IC,FORMAT_IQ,FORMAT_QUAL_bnd1,FORMAT_RASQ_bnd1,FORMAT_REF_bnd1,FORMAT_REFPAIR_bnd1,FORMAT_RP,FORMAT_RPQ,FORMAT_SR,FORMAT_SRQ,FORMAT_VF_bnd1,INFO_SVLEN,INFO_ANRP_bnd1,INFO_ANRPQ_bnd1,INFO_ANSR_bnd1,INFO_ANSRQ_bnd1,INFO_AS_bnd1,INFO_ASC_bnd1,INFO_ASQ_bnd1,INFO_ASRP_bnd1,INFO_ASSR,INFO_BA,INFO_BAQ,INFO_BASRP,INFO_BASSR,INFO_BMQ_bnd1,INFO_BMQN_bnd1,INFO_BMQX_bnd1,INFO_BQ_bnd1,INFO_BSC_bnd1,INFO_BSCQ_bnd1,INFO_BUM_bnd1,INFO_BUMQ_bnd1,INFO_BVF_bnd1,INFO_CAS_bnd1,INFO_CASQ_bnd1,INFO_CQ,INFO_EVENT,INFO_IC,INFO_INSRMP,INFO_INSRMRC,INFO_INSRMRO,INFO_INSRMRT,INFO_INSTAXID,INFO_IQ,INFO_MQ_bnd1,INFO_MQN,INFO_MQX,INFO_RAS_bnd1,INFO_RASQ_bnd1,INFO_REF_bnd1,INFO_REFPAIR_bnd1,INFO_RF,INFO_RP,INFO_RPQ,INFO_SB_bnd1,INFO_SC_bnd1,INFO_SR,INFO_SRQ,INFO_SVTYPE,INFO_VF_bnd1,INFO_CIPOS_l_bnd1,INFO_CIPOS_r_bnd1,bnd_pos_encoding_bnd1,CHROM_bnd2,POS_bnd2,ID_bnd2,REF_bnd2,ALT_bnd2,QUAL_bnd2,FORMAT_ANRP_bnd2,FORMAT_ANRPQ_bnd2,FORMAT_ANSR_bnd2,FORMAT_ANSRQ_bnd2,FORMAT_ASQ_bnd2,FORMAT_ASRP_bnd2,FORMAT_BQ_bnd2,FORMAT_BSC_bnd2,FORMAT_BSCQ_bnd2,FORMAT_BUM_bnd2,FORMAT_BUMQ_bnd2,FORMAT_BVF_bnd2,FORMAT_CASQ_bnd2,FORMAT_QUAL_bnd2,FORMAT_RASQ_bnd2,FORMAT_REF_bnd2,FORMAT_REFPAIR_bnd2,FORMAT_VF_bnd2,INFO_ANRP_bnd2,INFO_ANRPQ_bnd2,INFO_ANSR_bnd2,INFO_ANSRQ_bnd2,INFO_AS_bnd2,INFO_ASC_bnd2,INFO_ASQ_bnd2,INFO_ASRP_bnd2,INFO_BMQ_bnd2,INFO_BMQN_bnd2,INFO_BMQX_bnd2,INFO_BQ_bnd2,INFO_BSC_bnd2,INFO_BSCQ_bnd2,INFO_BUM_bnd2,INFO_BUMQ_bnd2,INFO_BVF_bnd2,INFO_CAS_bnd2,INFO_CASQ_bnd2,INFO_MQ_bnd2,INFO_RAS_bnd2,INFO_RASQ_bnd2,INFO_REF_bnd2,INFO_REFPAIR_bnd2,INFO_SB_bnd2,INFO_SC_bnd2,INFO_VF_bnd2,INFO_CIPOS_l_bnd2,INFO_CIPOS_r_bnd2,bnd_pos_encoding_bnd2,bptype
0,chr1,6558931,gridss0bb_1063o,G,[chr1:6559638[G,250.74,0,0.00,0,0.00,159.80,6,4,0.0,0,0,37.96,1,15.81,1,22.15,2,0.0,.,0,0.0,250.74,0.00,36,32,6,90.94,0,0.00,10,0,0,0.00,0,0.00,1,1X32N1X59D177M,159.80,6,4,0,0.0,0,0,60.00,60.0,60.0,37.96,1,15.81,1,22.15,2,0,0.0,250.74,gridss0bb_1063,0,0.0,0,0,0,0,0.0,57.63,36.0,60.0,1,0.00,36,32,0,6,90.94,0.000000,1X32N1X59D177M,0,0.00,BND,10,-16,17,4,chr1,6559638,gridss0bb_1063h,G,[chr1:6558931[G,250.74,0,0.00,0,0.00,0.00,6,39.27,1,17.12,1,22.15,1,0.0,250.74,159.80,40,24,10,0,0.00,0,0.00,1,1X3N1X28D70M,0.00,6,57.50,56.0,59.0,39.27,1,17.12,1,22.15,1,0,0.0,57.63,1,159.80,40,24,0.000000,1X3N1X98M,10,-17,16,4,44
1,chr1,6705183,gridss0bb_1096o,T,[chr8:130418562[T,137.01,0,0.00,6,137.01,0.00,0,0,0.0,0,0,66.15,0,0.00,3,66.15,1,0.0,.,0,0.0,137.01,0.00,52,26,0,0.00,6,137.01,6,0,0,0.00,6,137.01,0,1X9N1X,0.00,0,0,0,0.0,0,0,34.67,22.0,60.0,66.15,0,0.00,3,66.15,1,0,0.0,137.01,gridss0bb_1096,0,0.0,0,0,0,0,0.0,27.33,21.0,37.0,0,0.00,52,26,0,0,0.00,0.666667,1X9N1X42M,6,137.01,BND,6,-5,5,4,chr8,130418562,gridss0bb_1096h,A,[chr1:6705183[A,137.01,0,0.00,6,137.01,0.00,0,0.00,0,0.00,0,0.00,0,0.0,137.01,0.00,47,27,6,0,0.00,6,137.01,0,1X9N1X,0.00,0,0.00,0.0,0.0,0.00,0,0.00,0,0.00,0,0,0.0,27.33,0,0.00,47,27,0.333333,1X9N1X28M,6,-5,5,4,44
2,chr1,6709788,gridss0bb_1098o,T,[chr10:131665332[T,227.71,0,0.00,1,23.16,0.00,0,10,0.0,0,0,42.77,2,42.77,0,0.00,2,0.0,.,0,0.0,227.71,204.55,53,14,0,0.00,1,23.16,10,0,0,0.00,1,23.16,0,1X13N1X32M,0.00,0,10,0,0.0,0,0,60.00,60.0,60.0,42.77,2,42.77,0,0.00,2,0,0.0,227.71,gridss0bb_1098,0,0.0,0,0,0,0,0.0,54.50,39.0,60.0,1,204.55,53,14,0,0,0.00,0.076923,1X13N1X32M,1,23.16,BND,10,-7,7,4,chr10,131665332,gridss0bb_1098h,A,[chr1:6709788[A,227.71,0,0.00,0,0.00,204.55,0,204.69,9,182.55,1,22.15,2,0.0,227.71,0.00,39,30,10,0,0.00,0,0.00,1,1X13N1X,204.55,0,58.00,40.0,60.0,204.69,9,182.55,1,22.15,2,0,0.0,54.50,0,0.00,39,30,0.100000,1X13N1X18M,10,-7,7,4,44
3,chr1,6740259,gridss0bb_1113o,T,[chr17:16413723[T,292.34,0,0.00,0,0.00,272.96,0,12,0.0,0,0,158.80,6,158.80,0,0.00,

In [18]:
set(gridss_callset_bps['bptype']).union(set(ref_callset_bps['bptype']))

{'12', '21', '33', '44'}

In [19]:
gridss_callset_bps.shape

(197360, 143)

In [22]:
callset_chrom_combs = set(pd.concat([(gridss_callset_bps['CHROM_bnd1'] + '_' + gridss_callset_bps['CHROM_bnd2']), (ref_callset_bps['CHROM_bnd1'] + '_' + ref_callset_bps['CHROM_bnd2'])]))
callset_chrom_combs

{'0_chr1',
 '0_chr12',
 '0_chr15',
 '0_chr16',
 '0_chr17',
 '0_chr2',
 '0_chr22',
 '0_chr3',
 '0_chr5',
 '0_chr7',
 '0_chr8',
 'chr10_chr10',
 'chr10_chr11',
 'chr10_chr12',
 'chr10_chr13',
 'chr10_chr14',
 'chr10_chr15',
 'chr10_chr16',
 'chr10_chr17',
 'chr10_chr18',
 'chr10_chr19',
 'chr10_chr2',
 'chr10_chr20',
 'chr10_chr21',
 'chr10_chr22',
 'chr10_chr3',
 'chr10_chr4',
 'chr10_chr5',
 'chr10_chr6',
 'chr10_chr7',
 'chr10_chr8',
 'chr10_chr9',
 'chr10_chrX',
 'chr10_chrY',
 'chr11_chr11',
 'chr11_chr12',
 'chr11_chr13',
 'chr11_chr14',
 'chr11_chr15',
 'chr11_chr16',
 'chr11_chr17',
 'chr11_chr18',
 'chr11_chr19',
 'chr11_chr2',
 'chr11_chr20',
 'chr11_chr21',
 'chr11_chr22',
 'chr11_chr3',
 'chr11_chr4',
 'chr11_chr5',
 'chr11_chr6',
 'chr11_chr7',
 'chr11_chr8',
 'chr11_chr9',
 'chr11_chrM',
 'chr11_chrX',
 'chr11_chrY',
 'chr12_chr12',
 'chr12_chr13',
 'chr12_chr14',
 'chr12_chr15',
 'chr12_chr16',
 'chr12_chr17',
 'chr12_chr18',
 'chr12_chr19',
 'chr12_chr2',
 'chr12_chr20',


In [ ]:
wall_dist = 10000

unique_bptypes = set(gridss_callset_bps['bptype']).union(set(ref_callset_bps['bptype']))

distance_df = pd.DataFrame()

for bptype in unique_bptypes:
    gridss_callset_bp_bybptype = gridss_callset_bps[gridss_callset_bps['bptype'] == bptype]
    ref_callset_bp_bybptype = ref_callset_bps[ref_callset_bps['bptype'] == bptype]
    if gridss_callset_bp_bybptype.shape[0] == 0 or ref_callset_bp_bybptype.shape[0] == 0:
        print(f'No breakpoints of type {bptype} in one of the callsets, skipping matching')
        continue
    # Get breakend coordinates for GRIDSS callset
    gridss_bnd1_bybptype_pr, gridss_bnd2_bybptype_pr = get_bnd_prs(gridss_callset_bp_bybptype, padding=(0, 0), retain_cols=['bptype'])

    # Get breakend coordinates for reference callset
    ref_bnd1_bybptype_pr, ref_bnd2_bybptype_pr = get_bnd_prs(ref_callset_bp_bybptype, padding=(0, 0), retain_cols=['bptype'])

    gridss_bnd1_bybptype_pr_kn = gridss_bnd1_bybptype_pr.k_nearest(ref_bnd1_bybptype_pr, k=1).df
    gridss_bnd1_dist_to_same_type_ref = gridss_bnd1_bybptype_pr_kn[['ID', 'Start', 'Start_b', 'Distance', 'ID_b']]
    gridss_bnd1_dist_to_same_type_ref = gridss_bnd1_dist_to_same_type_ref.rename(columns={'Distance': 'INFO_BND1_DIST', 'Start': 'Start_bnd1', 'Start_b': 'Start_b_bnd1', 'ID_b': 'ID_b_bnd1'})
    gridss_bnd2_bybptype_pr_kn = gridss_bnd2_bybptype_pr.k_nearest(ref_bnd2_bybptype_pr, k=1).df
    gridss_bnd2_dist_to_same_type_ref = gridss_bnd2_bybptype_pr_kn[['Start', 'Start_b', 'Distance', 'ID_b']]
    gridss_bnd2_dist_to_same_type_ref = gridss_bnd2_dist_to_same_type_ref.rename(columns={'Distance': 'INFO_BND2_DIST', 'Start': 'Start_bnd2', 'Start_b': 'Start_b_bnd2', 'ID_b': 'ID_b_bnd2'})

    gridss_dist_bybptype = pd.concat([gridss_bnd1_dist_to_same_type_ref, gridss_bnd2_dist_to_same_type_ref], axis=1)

    gridss_unmatched_mask = (np.abs(gridss_dist_bybptype['INFO_BND1_DIST']) > wall_dist) | (np.abs(gridss_dist_bybptype['INFO_BND2_DIST']) > wall_dist)

    distance_df = pd.concat([distance_df, gridss_dist_bybptype])

print(distance_df)




In [ ]:
bp_array1 = gridss_callset_bps[['POS_bnd1', 'POS_bnd2']].to_numpy()
bp_array2 = ref_callset_bps[['POS_bnd1', 'POS_bnd2']].to_numpy()

tree = cKDTree(bp_array2)
dists, idx = tree.query(bp_array1, 1)
# nn_dist = dists[0][:, 1]

dists.shape

In [136]:
from scipy.spatial import cKDTree


In [ ]:
for chrom1 in gridss_callset_bps['CHROM_bnd1'].unique():
    for chrom2 in gridss_callset_bps['CHROM_bnd1'].unique():
        mask1 = (gridss_callset_bps['CHROM_bnd1'] == chrom1) & (gridss_callset_bps['CHROM_bnd2'] == chrom2)
        mask2 = (ref_callset_bps['CHROM_bnd1'] == chrom1) & (ref_callset_bps['CHROM_bnd2'] == chrom2)
        # print(gridss_callset_bps[mask1])
        # print(ref_callset_bps[mask2])

        bp_array1 = gridss_callset_bps[mask1][['POS_bnd1', 'POS_bnd2']].to_numpy()
        bp_array2 = ref_callset_bps[mask2][['POS_bnd1', 'POS_bnd2']].to_numpy()

        if bp_array1.shape[0] == 0 or bp_array2.shape[0] == 0:
            continue
        
        tree = cKDTree(bp_array2)
        dists, idx = tree.query(bp_array1, 1)
        print(idx)

        matched_gridss_bnds = gridss_callset_bps[mask1].reset_index(drop=True)
        matched_ref_bnds = ref_callset_bps[mask2].iloc[idx].reset_index(drop=True)

        matched_bnds = pd.concat([matched_gridss_bnds, matched_ref_bnds], axis=1)
        print(matched_bnds)

matched_bnds


In [ ]:
bp_array2.shape

In [ ]:
gridss_callset_bps

In [ ]:
distance_df[distance_df['ID_b_bnd1'] == distance_df['ID_b_bnd2']]

In [ ]:
        gridss_dist_bybptype[full_matches_mask]['ID_b_bnd1']

        gridss_callset_bp_bybptype = gridss_callset_bp_bybptype[gridss_callset_bp_bybptype['ID'].isin(ids_fully_unmatched)]
        ref_callset_bp_bybptype = ref_callset_bp_bybptype[~ref_callset_bp_bybptype['ID'].isin(matched_ids_b)]

        distance_df = pd.concat([distance_df, gridss_dist_bybptype[full_matches_mask]])




In [ ]:
no_matches_mask = (np.abs(distance_df['INFO_BND1_DIST']) > wall_dist) | (np.abs(distance_df['INFO_BND2_DIST']) > wall_dist)
full_matches_mask = ~no_matches_mask & (distance_df['ID_b_bnd1'] == distance_df['ID_b_bnd2'])

distance_df[~(no_matches_mask | full_matches_mask)]


In [ ]:
gridss_callset_bps[~gridss_callset_bps['INFO_EVENT'].isin(distance_df['ID'])]

In [ ]:
gridss_bnd1_pr_kn = gridss_bnd1_pr.k_nearest(ref_bnd1_pr, k=1).df
# gridss_bnd1_pr_kn = gridss_bnd1_pr_kn[(gridss_bnd1_pr_kn['Distance'] == 0) & (gridss_bnd1_pr_kn['bptype'] == gridss_bnd1_pr_kn['bptype_b'])]
gridss_bnd1_pr_kn = gridss_bnd1_pr_kn[(gridss_bnd1_pr_kn['bptype'] == gridss_bnd1_pr_kn['bptype_b'])]
gridss_bnd1_pr_kn

In [ ]:
# Join GRIDSS and reference coordinates to find overlapping regions
# 'left' join keeps all GRIDSS calls and matches them to reference where possible
# report_overlap=True adds an 'Overlap' column with the overlap length
joined_call_ref_bnd1_pr = gridss_bnd1_pr.join(ref_bnd1_pr, how='left', report_overlap=True).df
joined_call_ref_bnd2_pr = gridss_bnd2_pr.join(ref_bnd2_pr, how='left', report_overlap=True).df

# Calculate overlap fraction for first breakend coordinates
# Takes maximum of:
# 1. Overlap length divided by GRIDSS interval length
# 2. Overlap length divided by reference interval length
# This accounts for cases where one interval contains the other
joined_call_ref_bnd1_pr.loc[:, 'overlap_frac'] = np.maximum(
    joined_call_ref_bnd1_pr['Overlap'] / (joined_call_ref_bnd1_pr['End'] - joined_call_ref_bnd1_pr['Start']),
    joined_call_ref_bnd1_pr['Overlap'] / (joined_call_ref_bnd1_pr['End_b'] - joined_call_ref_bnd1_pr['Start_b'])
)

# Calculate overlap fraction for second breakend coordinates
# Uses same approach as first breakend
joined_call_ref_bnd2_pr.loc[:, 'overlap_frac'] = np.maximum(
    joined_call_ref_bnd2_pr['Overlap'] / (joined_call_ref_bnd2_pr['End'] - joined_call_ref_bnd2_pr['Start']),
    joined_call_ref_bnd2_pr['Overlap'] / (joined_call_ref_bnd2_pr['End_b'] - joined_call_ref_bnd2_pr['Start_b'])
)

# Find IDs where first breakend has good overlap with reference
# Requires:
# 1. >80% overlap between GRIDSS and reference coordinates
# 2. Matching breakend types between GRIDSS and reference
bnd1_overlapped_ids = joined_call_ref_bnd1_pr[
    (joined_call_ref_bnd1_pr['overlap_frac'] > 0.8) &\
    (joined_call_ref_bnd1_pr['bptype'] == joined_call_ref_bnd1_pr['bptype_b'])
]['ID'].unique()

# Find IDs where second breakend has good overlap with reference
# Uses same criteria as first breakend
bnd2_overlapped_ids = joined_call_ref_bnd2_pr[
    (joined_call_ref_bnd2_pr['overlap_frac'] > 0.8) &\
    (joined_call_ref_bnd2_pr['bptype'] == joined_call_ref_bnd2_pr['bptype_b'])
]['ID'].unique()

# Return arrays of IDs that have good overlap in both breakends
bnd1_overlapped_ids, bnd2_overlapped_ids


In [ ]:
bnd12_overlapped_ids = bnd1_overlapped_ids[np.isin(bnd1_overlapped_ids, bnd2_overlapped_ids)]
bnd12_overlapped_ids


In [ ]:
matching_ids = find_matching_breakpoint_ids(gridss_callset_bps, ref_callset_bps)
matching_ids


In [ ]:
gridss_callset_bps.loc[:, 'is_in_training'] = gridss_callset_bps['INFO_EVENT'].isin(matching_ids)
gridss_callset_bps.is_in_training.value_counts()

In [38]:
# Columns to remove from the final dataset
# cols_to_drop_approx = [
#     # Location columns
#     'CHROM', 'POS', 'Chromosome', 'Start', 'End',
    
#     # Reference/variant columns 
#     'REF', 'ALT',
    
#     # Format columns
#     'FORMAT_GT', 'FORMAT_VF',
    
#     # INFO fields with multiple values/string values
#     'INFO_SVLEN',
#     'INFO_ASC',
#     'INFO_BEID', 
#     'INFO_EVENT',
#     'INFO_HOMSEQ',
#     'INFO_MATEID',
#     'INFO_SC',
#     'INFO_SVTYPE',
#     'INFO_CIPOS',
#     'INFO_CIRPOS',
    
#     # Encoding column
#     'bnd_pos_encoding'
# ]

gridss_callset_bps_mult_val_cols = gridss_callset_bps.nunique() > 10
gridss_callset_bps_one_val_cols = gridss_callset_bps.nunique() == 1
gridss_callset_bps_obj_cols = gridss_callset_bps.dtypes == 'object'

gridss_callset_bps_cols_to_drop = gridss_callset_bps.columns[(gridss_callset_bps_mult_val_cols & gridss_callset_bps_obj_cols) | gridss_callset_bps_one_val_cols]

gridss_force_include = ['INFO_EVENT', 'bptype']

gridss_callset_bps_cols_to_drop = gridss_callset_bps_cols_to_drop[~gridss_callset_bps_cols_to_drop.isin(gridss_force_include)]

# cols_to_drop = []

# Create final dataset by dropping unnecessary columns
gridss_callset_final = gridss_callset_bps.drop(columns=gridss_callset_bps_cols_to_drop)

In [ ]:
gridss_callset_final

In [ ]:
gridss_callset_final.to_csv('saved/gridss.overlapped_bps.csv', index=False)

In [10]:

# ref_dels_bp_start = ref_callset_dels[['CHROM', 'POS', 'ID']]
# ref_dels_bp_start = ref_dels_bp_start.rename(columns={'CHROM': 'Chromosome'})
# ref_dels_bp_start.loc[:, 'Start'] = ref_dels_bp_start['POS'] - 100
# ref_dels_bp_start.loc[:, 'End'] = ref_dels_bp_start['POS'] + 101

# ref_dels_bp_end = ref_callset_dels[['INFO_CHR2', 'INFO_END', 'ID']]
# ref_dels_bp_end = ref_dels_bp_end.rename(columns={'INFO_CHR2': 'Chromosome'})
# ref_dels_bp_end.loc[:, 'Start'] = ref_dels_bp_end['INFO_END'] - 100
# ref_dels_bp_end.loc[:, 'End'] = ref_dels_bp_end['INFO_END'] + 101


In [11]:
# ref_dels_bp_start_pr = pr.PyRanges(ref_dels_bp_start)
# ref_dels_bp_end_pr = pr.PyRanges(ref_dels_bp_end)

In [31]:
# gridss_call_test = gridss_callset

# grsizes = gridss_call_test.groupby('INFO_EVENT').transform('size')
# gridss_call_test = gridss_call_test[grsizes == 2].sort_values(['INFO_EVENT', 'CHROM', 'POS'])
# gridss_call_test_start, gridss_call_test_end = gridss_call_test.iloc[::2].reset_index(drop=True), gridss_call_test.iloc[1::2].reset_index(drop=True)

In [32]:

# gridss_call_test_start = gridss_call_test_start[['CHROM', 'POS', 'ID']]
# gridss_call_test_start = gridss_call_test_start.rename(columns={'CHROM': 'Chromosome'})
# gridss_call_test_start.loc[:, 'Start'] = gridss_call_test_start['POS'] - 100
# gridss_call_test_start.loc[:, 'End'] = gridss_call_test_start['POS'] + 101

# gridss_call_test_end = gridss_call_test_end[['CHROM', 'POS', 'ID']]
# gridss_call_test_end = gridss_call_test_end.rename(columns={'CHROM': 'Chromosome'})
# gridss_call_test_end.loc[:, 'Start'] = gridss_call_test_end['POS'] - 100 
# gridss_call_test_end.loc[:, 'End'] = gridss_call_test_end['POS'] + 101


# gridss_call_test_end_pr = pr.PyRanges(gridss_call_test_end)
# gridss_call_test_start_pr = pr.PyRanges(gridss_call_test_start)

In [33]:
# gridss_call_test_start_pr = gridss_call_test_start_pr.join(ref_dels_bp_start_pr, how='left', report_overlap=True).df.reset_index(drop=True)
# gridss_call_test_start_pr.loc[:, 'overlap_frac'] = 2 * gridss_call_test_start_pr['Overlap'] / (
#     gridss_call_test_start_pr['End'] - gridss_call_test_start_pr['Start'] + gridss_call_test_start_pr['End_b'] - gridss_call_test_start_pr['Start_b']
# )

# # gridss_call_test_start_pr.loc[:, 'overlap_frac'].value_counts()

In [ ]:
# gridss_call_test_start_pr.sort_values(['ID', 'overlap_frac'], ascending=[True, False]).drop_duplicates(subset='ID', keep='first')[gridss_call_test_start_pr['overlap_frac'] > 0.8]